# Autonomous EEGNet decoder: nested Leave-One-Subject-Out and augmentation ablation

## 1. Research question and scope
This notebook implements an autonomous, fully blinded nested Leave-One-Subject-Out (LOSO) cross-validation protocol to evaluate an EEGNet-based 4-class motor imagery Brain-Computer Interface (BCI). 

The primary objective is twofold:
1. **Hyperparameter selection:** hyperparameters are selected within an inner cross-validation loop utilizing only the development subjects. The outer test subject is completely blinded and held out until the final evaluation.
2. **Compute-matched augmentation ablation:** To objectively evaluate the impact of training-only on-the-fly Gaussian augmentation (`Aug`) versus a standard training protocol (`NoAug`). Both conditions strictly share the same architecture, hyperparameter configuration (selected via mean performance across both conditions in the inner loop), initialization seed, and computational budget (batch size, examples per epoch, and updates per epoch).

**Disclaimer:** This is an external/clinical validation step on an existing dataset. While the evaluation loop is fully blinded, the dataset characteristics have been previously observed during the development history.

In [1]:
# Imports and central configuration
import os
import json
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import mne
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, 
                             f1_score, cohen_kappa_score, classification_report, confusion_matrix)

# Central Configuration Block
CONFIG = {
    'experiment_name': 'Nested_LOSO_Augmentation',
    'global_seed': 42,
    'data_dir': os.path.join('.', 'cleaned_data_1_40'),
    'output_dir': os.path.join('.', 'nested_loso_outputs'),
    'smoke_test': False, # If True, truncates data/epochs for testing logic
    
    # Data definitions
    'subjects': [f'A{i:02d}T' for i in range(1, 10)],
    'class_map': {'left': 1, 'right': 2, 'foot': 3, 'tongue': 4},
    'class_names': ['Left Hand', 'Right Hand', 'Foot', 'Tongue'],
    'labels': [0, 1, 2, 3],
    'fs': 250,
    'tmin': 0.5,
    'tmax': 3.5,
    'band': [1.0, 40.0],
    
    # Outer/Inner Protocol definitions
    'n_inner_splits': 3,
    
    # Hyperparameter Grid (Inner Selection)
    'grid': {
        'learning_rate': [1e-3, 5e-4],
        'dropout_rate': [0.5],
        'batch_size': [32]
    },
    
    # Training Budgets
    'inner_max_epochs': 60,
    'inner_patience': 15,
    'monitor_metric': 'val_loss',
    
    # Augmentation
    'aug_sigma': 0.05,     # Applied on Z-scored signals
    'aug_prob': 0.5        # Probability of augmenting a specific trial in a training batch
}

if CONFIG['smoke_test']:
    print("WARNING: SMOKE TEST MODE IS ACTIVE. Results will not be valid for final reporting.")
    CONFIG['inner_max_epochs'] = 2
    CONFIG['subjects'] = CONFIG['subjects'][:3]

# Ensure output directory exists
os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Library versions
with open(os.path.join(CONFIG['output_dir'], 'environment_versions.json'), 'w') as f:
    json.dump({'numpy': np.__version__, 'pandas': pd.__version__, 
               'mne': mne.__version__, 'tensorflow': tf.__version__}, f, indent=4)

def set_global_seeds(seed=CONFIG['global_seed']):
    """Enforces reproducibility across Python, NumPy, and TensorFlow."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_global_seeds()

## Data and annotation audit
This module audits the input data to ensure channel count, time samples, sampling rates, and label balance strictly follow the anticipated structure before proceeding.

In [ ]:
# Data and annotation audit
def audit_and_load_subject(subject_id, config=CONFIG):
    """
    Audits the subject's .fif file, derives dimensions dynamically, and applies per-epoch Z-scoring.
    Offline analysis restricted to exact epoch ranges.
    """
    file_path = os.path.join(config['data_dir'], f'{subject_id}_clean_1_40_raw.fif')
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing required data file: {file_path}")
        
    raw = mne.io.read_raw_fif(file_path, preload=True, verbose=False)
    
    # Check sampling frequency
    assert raw.info['sfreq'] == config['fs'], f"{subject_id} sfreq is {raw.info['sfreq']} != {config['fs']}"
    
    # Extract EEG only
    raw.pick('eeg')
    
    # Events explicitly mapped to avoid alphabetical issues
    events, event_dict = mne.events_from_annotations(raw, event_id=config['class_map'], verbose=False)
    
    epochs = mne.Epochs(raw, events, event_id=config['class_map'], 
                        tmin=config['tmin'], tmax=config['tmax'],
                        baseline=None, preload=True, verbose=False)
    
    X = epochs.get_data(copy=True).astype(np.float32)
    y = epochs.events[:, -1] - 1  # Convert to 0-indexed [0, 1, 2, 3]
    
    # Audit Checks
    n_trials, n_channels, n_samples = X.shape
    
    if np.isnan(X).any() or np.isinf(X).any():
        raise ValueError(f"NaNs or Infs detected in subject {subject_id} data.")
        
    var_check = np.var(X, axis=-1)
    if (var_check == 0).any():
        raise ValueError(f"Zero variance channels detected in subject {subject_id} before Z-score.")
        
    # Derive and declare shapes dynamically
    print(f"Audit {subject_id}: {n_trials} trials, {n_channels} channels, {n_samples} samples at {config['fs']}Hz.")
    
    # Trial balance audit
    class_counts = np.bincount(y, minlength=len(config['class_map']))
    print(f"   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): {class_counts.tolist()}")
    
    # Per-epoch, per-channel Z-score normalization
    # Explicitly declared as offline analysis on complete epochs
    X_mean = X.mean(axis=-1, keepdims=True)
    X_std = X.std(axis=-1, keepdims=True) + 1e-8
    X_zscored = (X - X_mean) / X_std
    
    # Adding trailing dimension for Keras Conv2D (Trials, Channels, Samples, 1)
    return X_zscored[..., np.newaxis], y

## Data loading and preprocessing
Loads all subjects into memory separated by participant identifier. 


In [ ]:
# Data loading and preprocessing
data_store = {}
X_shape = None

print("Starting strict data loading and audit...")
for sub in CONFIG['subjects']:
    X_sub, y_sub = audit_and_load_subject(sub, CONFIG)
    
    if CONFIG['smoke_test']:
        X_sub, y_sub = X_sub[:24], y_sub[:24]
        
    data_store[sub] = (X_sub, y_sub)
    
    if X_shape is None:
        X_shape = X_sub.shape[1:]
    else:
        assert X_sub.shape[1:] == X_shape, f"Shape mismatch in {sub}: {X_sub.shape[1:]} != {X_shape}"

CONFIG['input_shape'] = X_shape
print(f"\nData successfully loaded. Derived input shape for architecture: {CONFIG['input_shape']}")

def stack_subjects(subject_list, store):
    """Combines a specified list of subjects into a single dataset."""
    X_stacked = np.concatenate([store[s][0] for s in subject_list], axis=0)
    y_stacked = np.concatenate([store[s][1] for s in subject_list], axis=0)
    
    # Track the participant of origin for inner group k-fold
    groups = []
    for s in subject_list:
        groups.extend([s] * len(store[s][1]))
    
    return X_stacked, y_stacked, np.array(groups)

Starting strict data loading and audit...
Audit A01T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A02T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A03T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A04T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A05T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A06T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A07T: 288 trials, 22 channels, 751 samples at 250Hz.
   Class balance (0:Left, 1:Right, 2:Foot, 3:Tongue): [72, 72, 72, 72]
Audit A08T: 288 trials, 22 channels, 751 samples

## EEGNet Architecture
Implementation of the original compact CNN extracting spatiotemporal features from Z-scored EEG signals.

In [ ]:
#  EEGNet architecture
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Activation, Dropout, Conv2D, AveragePooling2D,
                                     SeparableConv2D, DepthwiseConv2D, BatchNormalization, Flatten, Input)
from tensorflow.keras.constraints import max_norm

def EEGNet_Model(nb_classes=4, input_shape=(22, 751, 1), dropoutRate=0.5, seed=None):
    """
    Standard EEGNet model for multiclass BCI decoding.
    Kernel constraints and regularizations included.
    """
    if seed is not None:
        tf.random.set_seed(seed)
        
    Chans, Samples, _ = input_shape
    input1 = Input(shape=input_shape)
    
    # Block 1: Temporal and Spatial Convolutions
    block1 = Conv2D(8, (1, 64), padding='same', use_bias=False)(input1)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=2,
                              depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropoutRate)(block1)

    # Block 2: Separable Convolution
    block2 = SeparableConv2D(16, (1, 16), use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropoutRate)(block2)

    flatten = Flatten()(block2)
    dense = Dense(nb_classes, kernel_constraint=max_norm(0.25))(flatten)
    softmax = Activation('softmax')(dense)
    
    return Model(inputs=input1, outputs=softmax)

## Training-only Gaussian Augmentation
On-the-fly augmentation utilizing `tf.data.Dataset`. This ensures the exact same compute budget (batch size, examples per epoch) as the `NoAug` condition, cleanly isolating the augmentation mechanism.

In [5]:
# 6. Training-only Gaussian augmentation
def create_tf_dataset(X, y, batch_size, augment=False, config=CONFIG, seed=42, shuffle=True):
    """
    Creates a tf.data.Dataset.
    If augment=True, applies independent Gaussian noise to `aug_prob` fraction of the batch 
    on-the-fly, strictly maintaining identical step/epoch counts as the NoAug baseline.
    """
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(X), seed=seed, reshuffle_each_iteration=True)
        
    dataset = dataset.batch(batch_size)
    
    if augment:
        # Separate generator for noise injection to ensure independence from shuffle
        noise_gen = tf.random.Generator.from_seed(seed + 100)
        mask_gen = tf.random.Generator.from_seed(seed + 200)
        
        def add_noise(x_batch, y_batch):
            batch_dim = tf.shape(x_batch)[0]
            # Generate Gaussian noise using the specific sigma for Z-scored signals
            noise = noise_gen.normal(shape=tf.shape(x_batch), mean=0.0, stddev=config['aug_sigma'])
            
            # Apply explicitly configurable probability mask
            mask_probs = mask_gen.uniform(shape=[batch_dim])
            mask = tf.cast(mask_probs < config['aug_prob'], tf.float32)
            mask = tf.reshape(mask, [-1, 1, 1, 1])
            
            augmented_x = x_batch + (noise * mask)
            return augmented_x, y_batch
            
        dataset = dataset.map(add_noise, num_parallel_calls=tf.data.AUTOTUNE)
        
    return dataset.prefetch(tf.data.AUTOTUNE)

## Training and checkpoint utilities
Functions to manage independent Keras sessions. The inner loop enforces early stopping exclusively on the internal validation folds, while the final outer fit utilizes fixed median epochs.

In [6]:

from tensorflow.keras.optimizers import Adam

def get_compiled_model(lr, drop, seed, config=CONFIG):
    """Generates a fresh, cleanly initialized model and optimizer to prevent weight transfer."""
    tf.keras.backend.clear_session()
    model = EEGNet_Model(nb_classes=len(config['class_map']), 
                         input_shape=config['input_shape'], 
                         dropoutRate=drop, 
                         seed=seed)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=Adam(learning_rate=lr), 
                  metrics=['accuracy'])
    return model

def train_inner_fold(X_tr, y_tr, X_val, y_val, lr, drop, bs, augment, seed, config=CONFIG):
    """Executes a single inner training run with early stopping."""
    set_global_seeds(seed)
    
    train_ds = create_tf_dataset(X_tr, y_tr, bs, augment=augment, config=config, seed=seed, shuffle=True)
    val_ds = create_tf_dataset(X_val, y_val, bs, augment=False, config=config, seed=seed, shuffle=False)
    
    model = get_compiled_model(lr, drop, seed, config)
    
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor=config['monitor_metric'], 
                                                  patience=config['inner_patience'],
                                                  restore_best_weights=True)]
    
    history = model.fit(train_ds, validation_data=val_ds, 
                        epochs=config['inner_max_epochs'], 
                        callbacks=callbacks, verbose=0)
    
    return model, history

def train_outer_final(X_dev, y_dev, lr, drop, bs, augment, epochs, seed, config=CONFIG):
    """Trains final model on all development subjects using the inner-selected fixed epoch median."""
    set_global_seeds(seed)
    
    train_ds = create_tf_dataset(X_dev, y_dev, bs, augment=augment, config=config, seed=seed, shuffle=True)
    
    model = get_compiled_model(lr, drop, seed, config)
    
    start_time = time.time()
    history = model.fit(train_ds, epochs=epochs, verbose=0) # NO early stopping
    total_time = time.time() - start_time
    
    updates_per_epoch = len(train_ds)
    total_updates = epochs * updates_per_epoch
    
    return model, history, total_time, total_updates

## Outer and inner split bonstruction
Generation of disjoint subsets guarantees that the test participant cannot interact with hyperparameter tuning. A manifest is generated to document these splits for reproducibility.

In [7]:
outer_inner_manifest = {}

# Pre-calculate the structure to ensure complete disjoint verification
for test_sub in CONFIG['subjects']:
    dev_subs = [s for s in CONFIG['subjects'] if s != test_sub]
    
    # Assert Disjunction
    assert test_sub not in dev_subs, "Leakage Alert: Test subject found in development pool."
    
    # Generate Inner Splits mapping
    gkf = GroupKFold(n_splits=CONFIG['n_inner_splits'])
    X_dummy, y_dummy, groups_dummy = stack_subjects(dev_subs, data_store)
    
    inner_manifest = []
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_dummy, y_dummy, groups_dummy)):
        train_groups = np.unique(groups_dummy[train_idx]).tolist()
        val_groups = np.unique(groups_dummy[val_idx]).tolist()
        
        # Verify inner isolation
        assert not set(train_groups).intersection(set(val_groups)), "Leakage Alert: Subject overlap in inner CV."
        inner_manifest.append({'train': train_groups, 'val': val_groups})
        
    outer_inner_manifest[test_sub] = {
        'development_subjects': dev_subs,
        'inner_splits': inner_manifest
    }

# Save split manifest
manifest_path = os.path.join(CONFIG['output_dir'], 'manifest_splits.json')
with open(manifest_path, 'w') as f:
    json.dump(outer_inner_manifest, f, indent=4)
    
print(f"Split manifest strictly verified and saved to {manifest_path}.")

Split manifest strictly verified and saved to .\nested_loso_outputs\manifest_splits.json.


## Inner hyperparameter selection  and outer-fold training
This section iterates over the outer folds, independently evaluating the grid on inner validation groups, calculating median epochs, and performing the final blind evaluation.

In [8]:
outer_evaluation_metrics = []
trial_level_predictions = []

grid_combinations = [(lr, drop, bs) 
                     for lr in CONFIG['grid']['learning_rate'] 
                     for drop in CONFIG['grid']['dropout_rate'] 
                     for bs in CONFIG['grid']['batch_size']]

total_trainings_predicted = len(CONFIG['subjects']) * CONFIG['n_inner_splits'] * len(grid_combinations) * 2 + len(CONFIG['subjects']) * 2
print(f"Total training fits scheduled: {total_trainings_predicted}")

for test_sub in CONFIG['subjects']:
    print(f"\n{'='*60}\nOuter Fold Test Subject: {test_sub}\n{'='*60}")
    dev_subs = outer_inner_manifest[test_sub]['development_subjects']
    
    X_dev, y_dev, groups_dev = stack_subjects(dev_subs, data_store)
    
    # ---------------------------------------------------------
    # INNER SEARCH
    # ---------------------------------------------------------
    inner_results = []
    for lr, drop, bs in grid_combinations:
        noaug_scores, aug_scores = [], []
        noaug_epochs, aug_epochs = [], []
        
        gkf = GroupKFold(n_splits=CONFIG['n_inner_splits'])
        for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(X_dev, y_dev, groups=groups_dev)):
            X_in_tr, y_in_tr = X_dev[tr_idx], y_dev[tr_idx]
            X_in_val, y_in_val = X_dev[val_idx], y_dev[val_idx]
            
            # Sub-Seed ensures NoAug and Aug start equally per split
            split_seed = CONFIG['global_seed'] + fold_idx 
            
            # --- Evaluate NoAug ---
            m_noaug, h_noaug = train_inner_fold(X_in_tr, y_in_tr, X_in_val, y_in_val, 
                                                lr, drop, bs, False, split_seed, CONFIG)
            p_noaug = np.argmax(m_noaug.predict(X_in_val, verbose=0), axis=1)
            bacc_noaug = balanced_accuracy_score(y_in_val, p_noaug)
            ep_noaug = np.argmin(h_noaug.history['val_loss']) + 1
            noaug_scores.append(bacc_noaug)
            noaug_epochs.append(ep_noaug)
            
            # --- Evaluate Aug ---
            m_aug, h_aug = train_inner_fold(X_in_tr, y_in_tr, X_in_val, y_in_val, 
                                            lr, drop, bs, True, split_seed, CONFIG)
            p_aug = np.argmax(m_aug.predict(X_in_val, verbose=0), axis=1)
            bacc_aug = balanced_accuracy_score(y_in_val, p_aug)
            ep_aug = np.argmin(h_aug.history['val_loss']) + 1
            aug_scores.append(bacc_aug)
            aug_epochs.append(ep_aug)

        # Config performance is explicitly the mean across BOTH conditions
        mean_bacc_total = np.mean(noaug_scores + aug_scores)
        
        # Explicit rules: median epoch > 0
        med_ep_noaug = max(1, int(np.median(noaug_epochs)))
        med_ep_aug = max(1, int(np.median(aug_epochs)))
        
        inner_results.append({
            'lr': lr, 'drop': drop, 'bs': bs,
            'mean_bacc_total': mean_bacc_total,
            'med_ep_noaug': med_ep_noaug,
            'med_ep_aug': med_ep_aug
        })
        
    best_config = max(inner_results, key=lambda x: x['mean_bacc_total'])
    print(f"  [Inner CV] Selected Shared Config: LR={best_config['lr']}, Drop={best_config['drop']}, BS={best_config['bs']}")
    print(f"             Target Epochs -> NoAug: {best_config['med_ep_noaug']}, Aug: {best_config['med_ep_aug']}")
    
    # Save inner results
    pd.DataFrame(inner_results).to_csv(os.path.join(CONFIG['output_dir'], f'inner_grid_{test_sub}.csv'), index=False)
    
    # ---------------------------------------------------------
    # FINAL OUTER EVALUATION (Completely blind)
    # ---------------------------------------------------------
    X_test, y_test, _ = stack_subjects([test_sub], data_store)
    
    for condition, augment, target_epochs in [
        ('NoAug', False, best_config['med_ep_noaug']),
        ('Aug', True, best_config['med_ep_aug'])
    ]:
        # Retrain on full dev dataset
        final_model, hist, t_time, t_updates = train_outer_final(
            X_dev, y_dev, best_config['lr'], best_config['drop'], best_config['bs'], 
            augment, target_epochs, seed=CONFIG['global_seed'], config=CONFIG
        )
        
        # Blind test evaluation
        preds_prob = final_model.predict(X_test, verbose=0)
        preds = np.argmax(preds_prob, axis=1)
        
        # Capture metrics
        acc = accuracy_score(y_test, preds)
        bacc = balanced_accuracy_score(y_test, preds)
        macro_f1 = f1_score(y_test, preds, average='macro')
        kappa = cohen_kappa_score(y_test, preds)
        
        outer_evaluation_metrics.append({
            'test_subject': test_sub,
            'condition': condition,
            'accuracy': acc,
            'balanced_accuracy': bacc,
            'macro_f1': macro_f1,
            'kappa': kappa,
            'n_trials': len(y_test),
            'training_time_s': t_time,
            'epochs': target_epochs,
            'updates': t_updates,
            'lr': best_config['lr']
        })
        
        # Save trial-level predictions
        for t_idx in range(len(y_test)):
            trial_level_predictions.append({
                'participant': test_sub,
                'trial_id': t_idx,
                'condition': condition,
                'seed': CONFIG['global_seed'],
                'true_class': y_test[t_idx],
                'pred_class': preds[t_idx],
                'prob_C0': preds_prob[t_idx, 0],
                'prob_C1': preds_prob[t_idx, 1],
                'prob_C2': preds_prob[t_idx, 2],
                'prob_C3': preds_prob[t_idx, 3]
            })
            
        # Checkpoint Model
        model_path = os.path.join(CONFIG['output_dir'], f"model_{test_sub}_{condition}.keras")
        final_model.save(model_path)
        
        print(f"  [Outer Test - {condition}] Acc: {acc:.3f}, B.Acc: {bacc:.3f}, Kappa: {kappa:.3f}")

# Flush comprehensive tables to disk
metrics_df = pd.DataFrame(outer_evaluation_metrics)
metrics_df.to_csv(os.path.join(CONFIG['output_dir'], 'outer_metrics.csv'), index=False)

preds_df = pd.DataFrame(trial_level_predictions)
preds_df.to_csv(os.path.join(CONFIG['output_dir'], 'trial_predictions.csv'), index=False)

Total training fits scheduled: 126

Outer Fold Test Subject: A01T

  [Inner CV] Selected Shared Config: LR=0.001, Drop=0.5, BS=32
             Target Epochs -> NoAug: 58, Aug: 58
  [Outer Test - NoAug] Acc: 0.618, B.Acc: 0.618, Kappa: 0.491
  [Outer Test - Aug] Acc: 0.611, B.Acc: 0.611, Kappa: 0.481

Outer Fold Test Subject: A02T
  [Inner CV] Selected Shared Config: LR=0.001, Drop=0.5, BS=32
             Target Epochs -> NoAug: 56, Aug: 56
  [Outer Test - NoAug] Acc: 0.444, B.Acc: 0.444, Kappa: 0.259
  [Outer Test - Aug] Acc: 0.434, B.Acc: 0.434, Kappa: 0.245

Outer Fold Test Subject: A03T
  [Inner CV] Selected Shared Config: LR=0.001, Drop=0.5, BS=32
             Target Epochs -> NoAug: 55, Aug: 55
  [Outer Test - NoAug] Acc: 0.681, B.Acc: 0.681, Kappa: 0.574
  [Outer Test - Aug] Acc: 0.681, B.Acc: 0.681, Kappa: 0.574

Outer Fold Test Subject: A04T
  [Inner CV] Selected Shared Config: LR=0.001, Drop=0.5, BS=32
             Target Epochs -> NoAug: 55, Aug: 55
  [Outer Test - NoAug] Acc

## Per-Participant results and aggregate and paired descriptive analyses


In [9]:
import warnings
warnings.filterwarnings('ignore') # Filter sklearn undefined metrics warning for small smoke batches

# Load exported data
res_df = pd.read_csv(os.path.join(CONFIG['output_dir'], 'outer_metrics.csv'))
preds_df = pd.read_csv(os.path.join(CONFIG['output_dir'], 'trial_predictions.csv'))

print("\n--- PER-PARTICIPANT BALANCED ACCURACY ---")
pivot_res = res_df.pivot(index='test_subject', columns='condition', values='balanced_accuracy')
pivot_res['Diff (Aug - NoAug)'] = pivot_res['Aug'] - pivot_res['NoAug']
print(pivot_res.round(3))

print("\n--- AGGREGATE SUBJECT-LEVEL STATISTICS ---")
# Properly aggregating average metrics across participants, NOT pooled predictions
agg_stats = res_df.groupby('condition')[['accuracy', 'balanced_accuracy', 'macro_f1', 'kappa']].agg(['mean', 'std'])
print(agg_stats.round(3))

print("\n--- POOLED CLASSIFICATION REPORT (NoAug) ---")
preds_noaug = preds_df[preds_df['condition'] == 'NoAug']
print(classification_report(preds_noaug['true_class'], preds_noaug['pred_class'], 
                            labels=CONFIG['labels'], target_names=CONFIG['class_names']))

print("\n--- POOLED CLASSIFICATION REPORT (Aug) ---")
preds_aug = preds_df[preds_df['condition'] == 'Aug']
print(classification_report(preds_aug['true_class'], preds_aug['pred_class'], 
                            labels=CONFIG['labels'], target_names=CONFIG['class_names']))

print("\n--- ABSOLUTE CONFUSION MATRICES ---")
print("NoAug Matrix:")
print(confusion_matrix(preds_noaug['true_class'], preds_noaug['pred_class'], labels=CONFIG['labels']))
print("\nAug Matrix:")
print(confusion_matrix(preds_aug['true_class'], preds_aug['pred_class'], labels=CONFIG['labels']))

print("\nNote on Statistical Inference: Paired differences calculated above. Treating non-independent cross-validation folds strictly as completely independent samples for p-values is discouraged. Descriptive pairing preserves structural reality.")


--- PER-PARTICIPANT BALANCED ACCURACY ---
condition       Aug  NoAug  Diff (Aug - NoAug)
test_subject                                  
A01T          0.611  0.618              -0.007
A02T          0.434  0.444              -0.010
A03T          0.681  0.681               0.000
A04T          0.455  0.417               0.038
A05T          0.479  0.469               0.010
A06T          0.528  0.576              -0.049
A07T          0.562  0.559               0.003
A08T          0.618  0.604               0.014
A09T          0.611  0.611               0.000

--- AGGREGATE SUBJECT-LEVEL STATISTICS ---
          accuracy        balanced_accuracy        macro_f1         kappa  \
              mean    std              mean    std     mean    std   mean   
condition                                                                   
Aug          0.553  0.085             0.553  0.085    0.542  0.088  0.404   
NoAug        0.553  0.090             0.553  0.090    0.540  0.097  0.404   

          

## Limitations and reproducibility checklist

### Limitations
* **Development history:** As this exact dataset format was observed and explored in earlier notebook iterations, this test, while execution-blinded, acts as a retrospective validation rather than a purely pristine unseen clinical trial.
* **Physiological attribution:** Output metrics summarize classification capacity but do not inherently prove localized desynchronization without subsequent integrated attribution mapping (e.g., Saliency, SHAP).

### Reproducibility and leakage-control checklist
- [x] External test participant completely held out of hyperparameter search.
- [x] Internal split strictly built on disjoint participant datasets (GroupKFold).
- [x] Z-score strictly calculated on individual 3.0s epoch sequences locally.
- [x] Random seeds enforced globally, isolating training augmentation shuffles.
- [x] Network topology fully re-instantiated and compiled for every fold without transferring intermediate weights.
- [x] Outer training restricted from tracking test metrics for early stopping.
- [x] Central configuration dictates all immutable architectural and experimental budgets.

### Results and Discussion: nested LOSO evaluation

The nested leave-one-subject-out (LOSO) evaluation assessed four-class motor-imagery EEG decoding using EEGNet across nine participants. Hyperparameter selection was performed through an inner subject-wise cross-validation procedure, while each outer test participant was excluded from model selection and final training decisions.

The results show nearly identical aggregate performance for the two training conditions. Mean accuracy and balanced accuracy were both 55.3% for NoAug and Aug. Mean macro-F1 increased only slightly, from 54.0% to 54.2%, while Cohen's kappa remained approximately 0.404 in both conditions. Overall, these results do not indicate a substantial average benefit from Gaussian-noise augmentation at the tested noise level.

The subject-level analysis reveals a different pattern. Balanced accuracy ranged from 41.7% to 68.1% across participants, demonstrating meaningful variability in cross-subject decoding performance. Augmentation improved four participants, reduced performance in three, and produced no change in two. The largest observed improvement was 3.8 percentage points for A04T, whereas the largest decrease was approximately 4.8 percentage points for A06T.

These findings suggest that the effect of augmentation is participant-dependent rather than consistently beneficial. Although small changes may be relevant in specific applications, the aggregate results do not support a clear overall advantage for the current augmentation strategy.

The pooled classification reports also show broadly similar performance across the four classes. For example, the F1-score for Tongue remained 0.60 under both conditions, while the Right Hand F1-score increased slightly from 0.54 to 0.55 with augmentation. Such class-level differences are small and should not be interpreted as evidence of a systematic improvement.

In summary, EEGNet achieved moderate cross-subject decoding performance, with greater variation across participants than the average difference between the augmentation conditions. These findings highlight the difficulty of generalizing motor-imagery EEG models to unseen individuals and suggest that improvements in subject-independent robustness may require approaches beyond the tested Gaussian-noise augmentation.

**Statistical caution:** the reported aggregate differences are descriptive. A formal paired statistical test should be calculated using the full-precision outer-fold results before making claims about statistical significance. Even if a paired test is non-significant, this does not establish equivalence between the two methods, particularly with only nine participants.